<a href="https://colab.research.google.com/github/TKhahahah/Data_Mining_FinalProject/blob/main/DTM_PJ1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load Data

In [42]:
import pandas as pd
import os, sys
import numpy as np
import copy
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pickle
from dateutil.relativedelta import relativedelta
import random

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
X_index = pd.read_csv('/content/drive/MyDrive/KKU3/X_variable(Index)_operation_v4.csv')
Y_rainfall = pd.read_csv('/content/drive/MyDrive/KKU3/Y_variable(Rainfall).csv')
# Convert 'DATE' column in Y_df to datetime objects if it's not already
X_index['DATE'] = pd.to_datetime(X_index['DATE'])

# Format the dates in Y_df as 'YYYY-MM-01'
X_index['DATE'] = X_index['DATE'].dt.strftime('%Y-%m-01')



node = pd.read_csv('/content/drive/MyDrive/KKU3/Node_table_TMD.csv')


path_feature = os.path.join("/content/drive/MyDrive/KKU3/lookup_table_reanalysis_v5.csv")
feature = pd.read_csv(path_feature, header= "infer")



In [4]:
X_index.head()

,DATE,MEIV2,RMM_AMPLITUDE,PHASE,BSISO1,BSISO1-Phase,DMI,ONI,PDO,SWM,NE
0,1981-01-01,0.35,1.79134,7.0,0.000,0.0,NaN,NaN,NaN,NaN,NaN
1,1981-02-01,0.23,2.04364,7.0,0.000,0.0,NaN,NaN,NaN,NaN,NaN
2,1981-03-01,0.33,3.16988,3.0,0.000,0.0,NaN,NaN,NaN,NaN,NaN
3,1981-04-01,0.42,3.22159,4.0,0.000,0.0,NaN,NaN,NaN,NaN,NaN
4,1981-05-01,0.24,0.00000,1.0,1.467,8.0,NaN,NaN,NaN,NaN,NaN


In [5]:
X_index.columns

Index(['DATE', 'MEIV2', 'RMM_AMPLITUDE', 'PHASE', 'BSISO1', 'BSISO1-Phase',
       'DMI', 'ONI', 'PDO', 'SWM', 'NE'],
      dtype='object')

In [6]:
Y_rainfall.head()

,DATE,300201,300202,303201,303301,310201,327202,327301,327501,328201,...,566202,567201,568301,568401,568501,568502,570201,580201,581301,583201
0,1961-01-01,0.5,13.1,0.4,NaN,NaN,NaN,NaN,1.8,0.4,...,NaN,71.2,NaN,NaN,58.5,NaN,NaN,NaN,NaN,193.6
1,1961-02-01,1.1,4.4,5.7,NaN,NaN,NaN,NaN,33.3,2.4,...,NaN,89.9,NaN,NaN,13.2,NaN,NaN,NaN,NaN,76.7
2,1961-03-01,7.6,0.0,37.7,NaN,NaN,NaN,NaN,22.5,46.2,...,NaN,62.1,NaN,NaN,5.4,NaN,NaN,NaN,NaN,24.5
3,1961-04-01,94.6,27.1,118.8,NaN,NaN,NaN,NaN,65.2,58.6,...,NaN,124.1,NaN,NaN,154.9,NaN,NaN,NaN,NaN,42.3
4,1961-05-01,223.9,189.5,257.9,NaN,NaN,NaN,NaN,249.8,240.9,...,NaN,219.7,NaN,NaN,114.8,NaN,NaN,NaN,NaN,80.7


In [7]:
Y_rainfall.columns

Index(['DATE', '300201', '300202', '303201', '303301', '310201', '327202',
       '327301', '327501', '328201',
       ...
       '566202', '567201', '568301', '568401', '568501', '568502', '570201',
       '580201', '581301', '583201'],
      dtype='object', length=133)

In [8]:
node.head()

,Node,Lat,Long,cluster,quality,lag
0,300201,19.299814,97.972734,7.0,1,NaN
1,300202,18.176442,97.930608,7.0,1,NaN
2,303201,19.961219,99.881294,7.0,1,NaN
3,303301,19.871839,99.779203,7.0,1,NaN
4,310201,19.192933,99.883650,NaN,0,NaN


In [9]:
feature.head()

,Cluster,Node,Lat,Long,quality,lag
0,1,SWM,0.0,101.0,0,0
1,1,DMI,0.0,80.0,1,1
2,1,PDO,0.0,165.0,1,1
3,1,ONI,0.0,180.0,0,0
4,1,NE,10.0,110.0,1,0


# Cleansing

In [10]:
algo='LSTM_V23_11'

num_epoch = 50
window_size = 24  #Window size
horizon = 12   #Prediction horizon
TBPTT_K = 24

learn_rate =0.001
weight_decay =1e-4
cluster = 6 #choose cluster of the HII stations
k_fold_num=2
percentile_loss =0.95

# PARAMETERS FOR SMALL DATA
noise_level = 0.05  # Standard deviation of noise (5% of signal)

min_epochs = 20
patience = 100
LOG_EVERY = 20


hidden_size_list =[128]
num_layers_list =[2]
drop_out_list=[0.4]



In [11]:
Y_rainfall = Y_rainfall.rename(columns={'code': 'Node'})

Y_rainfall_clean = Y_rainfall.copy()
num_feature = len(feature[feature['Cluster']==cluster])

In [12]:

def apply_time_lag(df, column_name, lag):
    # Create a new column name with the lag suffix
    new_column_name = f"{column_name}_lag{lag}"

    # Shift the column values by the specified lag
    df[new_column_name] = df[column_name].shift(lag)

    return df

# Define the time lags for each climate index

time_lags = {

    'MEIV2': [1, 2, 3],
    'RMM_AMPLITUDE': [1],
    'PHASE': [1],
    'PDO': [1, 2],
    'ONI': [1, 2],
    'DMI': [1, 2],
    'BSISO1': [1],
    'BSISO1-Phase': [1],
    'SWM': [1],

}




# Apply time lags to each specified column in df_merge_type
for column, lag in time_lags.items():
      if column in X_index.columns:
            for lag_item in lag:
                X_index = apply_time_lag(X_index, column, lag_item)
      else:
        print(f"Warning: Column '{column}' not found in df_merge_type. Skipping.")


## start clean

In [13]:
HQ_date_get_month_df  =  Y_rainfall.copy()
Y_rainfall_clean =  Y_rainfall.copy()
HQ_date_get_month_df['DATE'] = pd.to_datetime(Y_rainfall['DATE']).dt.month

month_avg_list = []
for month in range(1, 12 + 1):
    # Filter the data for the current year
    HQ_data = HQ_date_get_month_df[HQ_date_get_month_df['DATE'] == month]

    # Average rainfall
    av_r = HQ_data.iloc[:, 1:].median()

    # Create a dictionary to hold the data for this year
    month_dict = {'MONTH': month}
    month_dict.update(av_r)

    # Append the data for this year to the list
    month_avg_list.append(month_dict)

HQ_month_avg = pd.DataFrame(month_avg_list)

for i in range(1, len(Y_rainfall_clean.columns)):
    for j in range(0, len(Y_rainfall_clean)):
        if(np.isnan(Y_rainfall_clean.iloc[j, i])):
             Y_rainfall_clean.iloc[j, i] = HQ_month_avg.iloc[pd.to_datetime(Y_rainfall_clean.iloc[:, 0]).dt.month[j]-1, i]

In [14]:
Y_rainfall_clean['DATE'].max()

'2025-03-01'

In [15]:
start_training_date = '1982-01-01'


X_index['DATE'] = pd.to_datetime(X_index['DATE'])
Y_rainfall_clean['DATE'] = pd.to_datetime(Y_rainfall_clean['DATE'])
max_date_X = X_index['DATE'].max()
max_date_Y = Y_rainfall_clean['DATE'].max()
# Minimum date possible

print("Maximum date possible : ",start_training_date)


# Maximum date possible
max_date_possible = min([max_date_X,max_date_Y])
print("Maximum date possible : ",max_date_possible)

#Make condition
con_date_X = (X_index['DATE'] >= start_training_date) & (X_index['DATE'] <= max_date_possible)
con_date_Y = (Y_rainfall_clean['DATE'] >= start_training_date) & (Y_rainfall_clean['DATE'] <= max_date_possible)

#Final select
X_index_interval_date = X_index.loc[con_date_X,:]
Y_rainfall_clean = Y_rainfall_clean.loc[con_date_Y,:]


Y_rainfall_interval_date = copy.deepcopy(Y_rainfall_clean)

Maximum date possible :  1982-01-01
Maximum date possible :  2025-03-01 00:00:00


In [16]:
start_training_date = Y_rainfall_clean['DATE'].min()

X_index['DATE'] = pd.to_datetime(X_index['DATE'])
Y_rainfall_clean['DATE'] = pd.to_datetime(Y_rainfall_clean['DATE'])
max_date_X = X_index['DATE'].max()
max_date_Y = Y_rainfall_clean['DATE'].max()
# Minimum date possible

print("Maximum date possible : ",start_training_date)


# Maximum date possible
max_date_possible = min([max_date_X,max_date_Y])
print("Maximum date possible : ",max_date_possible)

#Make condition
con_date_X = (X_index['DATE'] >= start_training_date) & (X_index['DATE'] <= max_date_possible)
con_date_Y = (Y_rainfall_clean['DATE'] >= start_training_date) & (Y_rainfall_clean['DATE'] <= max_date_possible)

#Final select
X_index_interval_date = X_index.loc[con_date_X,:]
Y_rainfall_clean = Y_rainfall_clean.loc[con_date_Y,:]


Y_rainfall_interval_date = copy.deepcopy(Y_rainfall_clean)

Maximum date possible :  1982-01-01 00:00:00
Maximum date possible :  2025-03-01 00:00:00


In [17]:
#Count null value of each indexs
print("Number of index columns which contain null value : ",len(X_index_interval_date.isnull().sum()[X_index_interval_date.isnull().sum()>0]), sep="")
print("Number of rainfall station which contain null value : ",len(Y_rainfall_interval_date.isnull().sum()[Y_rainfall_interval_date.isnull().sum()>0]), sep="")

Number of index columns which contain null value : 0
Number of rainfall station which contain null value : 0


In [18]:
def convert_columns_to_index_columns(df,col_name):
    df.loc[:,col_name] = pd.to_datetime(df[col_name])
    df = df.set_index(col_name, inplace=False)
    return df

In [19]:
X_index_df_ready = convert_columns_to_index_columns(X_index_interval_date, 'DATE')
Y_rainfall_df_ready = convert_columns_to_index_columns(Y_rainfall_interval_date, 'DATE')

In [20]:
print("nrow of X : {}".format(X_index_df_ready.shape[0]))
print("nrow of Y : {}".format(X_index_df_ready.shape[0]))

nrow of X : 519
nrow of Y : 519


In [21]:

#scaler_X = MinMaxScaler()
scaler_X = StandardScaler()

X_normalize_data = scaler_X.fit_transform(X_index_df_ready)
X_index_normalized_df = pd.DataFrame(X_normalize_data, columns=X_index_df_ready.columns)

scaler_Y = StandardScaler()

Y_normalize_data = scaler_Y.fit_transform(Y_rainfall_df_ready)
Y_rainfall_normalized_df = pd.DataFrame(Y_normalize_data, columns=Y_rainfall_df_ready.columns)

In [22]:
#Get date range from original files because after normalization index will reset
og_date_range = X_index_interval_date['DATE']
X_index_normalized_df.set_index(og_date_range, inplace=True)
Y_rainfall_normalized_df.set_index(og_date_range, inplace=True)

n_feature_climate = X_index_normalized_df.shape[1]
feature_climate = X_index_normalized_df.columns
print("Number of climate index feature : ",n_feature_climate)
print("All climate index : \n",feature_climate)

Number of climate index feature :  24
All climate index : 
 Index(['MEIV2', 'RMM_AMPLITUDE', 'PHASE', 'BSISO1', 'BSISO1-Phase', 'DMI',
       'ONI', 'PDO', 'SWM', 'NE', 'MEIV2_lag1', 'MEIV2_lag2', 'MEIV2_lag3',
       'RMM_AMPLITUDE_lag1', 'PHASE_lag1', 'PDO_lag1', 'PDO_lag2', 'ONI_lag1',
       'ONI_lag2', 'DMI_lag1', 'DMI_lag2', 'BSISO1_lag1', 'BSISO1-Phase_lag1',
       'SWM_lag1'],
      dtype='object')


In [23]:
X_index_normalized_df['Month_sin'] = np.sin(2 * np.pi * X_index_normalized_df.index.month / 12)
X_index_normalized_df['Month_cos'] = np.cos(2 * np.pi * X_index_normalized_df.index.month / 12)

In [24]:
station_id_list = Y_rainfall_normalized_df.columns
list_table = []
for code_st in station_id_list:
  #print(code_st)

  #print(static_value)
  eindices_df = X_index_normalized_df.copy()



  eindices_df[code_st] = Y_rainfall_normalized_df[code_st]

  list_table.append(eindices_df)

In [25]:
n_feature_climate = X_index_normalized_df.shape[1]
feature_climate = X_index_normalized_df.columns
print("Number of climate index feature : ",n_feature_climate)
print("All climate index : \n",feature_climate)

Number of climate index feature :  26
All climate index : 
 Index(['MEIV2', 'RMM_AMPLITUDE', 'PHASE', 'BSISO1', 'BSISO1-Phase', 'DMI',
       'ONI', 'PDO', 'SWM', 'NE', 'MEIV2_lag1', 'MEIV2_lag2', 'MEIV2_lag3',
       'RMM_AMPLITUDE_lag1', 'PHASE_lag1', 'PDO_lag1', 'PDO_lag2', 'ONI_lag1',
       'ONI_lag2', 'DMI_lag1', 'DMI_lag2', 'BSISO1_lag1', 'BSISO1-Phase_lag1',
       'SWM_lag1', 'Month_sin', 'Month_cos'],
      dtype='object')


In [26]:
n_feature_climate

26

In [27]:
feature_climate

Index(['MEIV2', 'RMM_AMPLITUDE', 'PHASE', 'BSISO1', 'BSISO1-Phase', 'DMI',
       'ONI', 'PDO', 'SWM', 'NE', 'MEIV2_lag1', 'MEIV2_lag2', 'MEIV2_lag3',
       'RMM_AMPLITUDE_lag1', 'PHASE_lag1', 'PDO_lag1', 'PDO_lag2', 'ONI_lag1',
       'ONI_lag2', 'DMI_lag1', 'DMI_lag2', 'BSISO1_lag1', 'BSISO1-Phase_lag1',
       'SWM_lag1', 'Month_sin', 'Month_cos'],
      dtype='object')

In [28]:
X_index_normalized_df

,MEIV2,RMM_AMPLITUDE,PHASE,BSISO1,BSISO1-Phase,DMI,ONI,PDO,SWM,NE,...,PDO_lag2,ONI_lag1,ONI_lag2,DMI_lag1,DMI_lag2,BSISO1_lag1,BSISO1-Phase_lag1,SWM_lag1,Month_sin,Month_cos
DATE,,,,,,,,,,,,,,,,,,,,,
1982-01-01,-0.187714,0.714459,-1.386540,-0.948166,-0.832762,-0.023677,-0.140501,0.366424,1.469381,1.793781,...,0.748840,-0.324733,-0.452551,-0.523783,-0.532089,-0.948166,-0.832762,1.136801,5.000000e-01,8.660254e-01
1982-02-01,-0.102040,0.281839,1.174365,-0.948166,-0.832762,1.265857,-0.042981,0.316895,0.837625,1.758152,...,0.634039,-0.140014,-0.324464,-0.021566,-0.522618,-0.948166,-0.832762,1.466406,8.660254e-01,5.000000e-01
1982-03-01,-1.056691,0.443087,0.747548,-0.948166,-0.832762,0.981589,-0.074650,0.177455,0.424145,1.158306,...,0.361165,-0.042503,-0.139761,1.268030,-0.020539,-0.948166,-0.832762,0.835332,1.000000e+00,6.123234e-17
1982-04-01,-0.261148,0.738476,1.601183,-0.948166,-0.832762,1.140032,0.200470,-0.098664,0.264021,1.416414,...,0.311641,-0.074169,-0.042259,0.983748,1.268705,-0.948166,-0.832762,0.422298,8.660254e-01,-5.000000e-01
1982-05-01,-0.579365,-0.930963,-0.532905,0.774939,-0.170381,1.531242,0.516375,-0.382280,-0.468001,0.380173,...,0.172211,0.200925,-0.073923,1.142199,0.984501,-0.948166,-0.832762,0.262348,5.000000e-01,-8.660254e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-01,0.118264,1.033203,-0.106088,-0.948166,-0.832762,-1.326749,-0.358461,-1.858967,-0.317086,0.487085,...,-1.454253,-0.210207,-0.189716,-1.170573,0.467312,1.437983,1.485571,-1.284616,-5.000000e-01,8.660254e-01
2024-12-01,0.399764,1.280297,0.320730,-0.948166,-0.832762,-1.191502,-0.533606,-0.917404,0.952684,0.793703,...,-1.731047,-0.357952,-0.209948,-1.324702,-1.169232,-0.948166,-0.832762,-0.318132,-2.449294e-16,1.000000e+00
2025-01-01,0.509916,1.073795,-0.532905,-0.948166,-0.832762,-1.005699,-0.629968,-0.503890,1.974034,1.132187,...,-1.864045,-0.533081,-0.357680,-1.189448,-1.323318,-0.948166,-0.832762,0.950266,5.000000e-01,8.660254e-01


In [29]:
Y_normalize_data

array([[-1.00118725, -0.98116192, -0.95348275, ..., -0.95299072,
        -1.14774134, -0.87020789],
       [-1.0195053 , -0.98425392, -0.99367516, ..., -0.9326291 ,
        -1.1459257 , -0.83825998],
       [-1.0195053 , -0.98425392, -0.9781623 , ..., -0.94580427,
        -0.81003255, -0.88249555],
       ...,
       [-1.0195053 , -0.98425392, -0.99367516, ..., -0.26788208,
        -0.08256666,  0.78985493],
       [-1.01275655, -0.57817157, -0.77931565, ..., -0.52779218,
        -0.39969821, -0.7362724 ],
       [-0.98094098, -0.98425392, -0.78354643, ..., -0.64277544,
        -0.59276112, -0.8337545 ]])

#Train-Test Split


In [32]:
#Train interval

start_interval_train1 =  pd.to_datetime(min(Y_rainfall_normalized_df.index))
#maximum_interval_train = max(Y_rainfall_normalized_df.index[0:-6])


end_interval_train1 = start_interval_train1 + relativedelta(months=37*12+11)


#Test1 interval
start_interval_val1 = start_interval_train1 + relativedelta(months=36*12)
end_interval_val1 =  start_interval_train1 + relativedelta(months=38*12+11)



#Test1 interval
start_interval_test1 = start_interval_train1 + relativedelta(months=37*12)
end_interval_test1 = start_interval_train1 + relativedelta(months=39*12+11)
#end_interval_test = start_interval_test + relativedelta(months=backward_months+17)
#Display
print("############################################################")
print("TRAINING Fold 1")
print('Start interval date of training set: ', start_interval_train1)
#print('Maximum interval date : ', maximum_interval_train)
print('End interval date of training set :', end_interval_train1)

print("############################################################")
print("VALIDATION Fold 1")
print('Start interval date of testing set: ', start_interval_val1)
print('End interval date of testing set :', end_interval_val1)
print("############################################################")



print("############################################################")
print("TESTING Fold 1")
print('Start interval date of testing set: ', start_interval_test1)
print('End interval date of testing set :', end_interval_test1)
print("############################################################")
#if cluster in [4 ,5,6,7, 8,9 ,10,11,12]:
#    start_interval_train1 = start_interval_val1 +  relativedelta(months=1)

#    end_interval_train2 = start_interval_train1 +  relativedelta(months=4*12+12)


    #Train interval
#    start_interval_val2 = start_interval_train1 +  relativedelta(months=3*12-1)
#    end_interval_val2 = start_interval_train1 +  relativedelta(months=5*12+10)



    #Test2 interval
#    start_interval_test2 = start_interval_train1 +  relativedelta(months=3*12+9)
#    end_interval_test2 = start_interval_train1 +  relativedelta(months=6*12+11-3)

#else:
start_interval_train1 = start_interval_train1

end_interval_train2 = start_interval_train1 + relativedelta(months=40*12)


    #Train interval
start_interval_val2 = start_interval_train1  + relativedelta(months=39*12)
end_interval_val2 = start_interval_train1 + relativedelta(months=41*12+11)



    #Test2 interval
start_interval_test2 = start_interval_train1 + relativedelta(months=40*12)
end_interval_test2 = start_interval_train1 + relativedelta(months=42*12+11)


#Display
print("############################################################")
print("TRAINING Fold 2")
print('Start interval date of training set: ', start_interval_train1)
#print('Maximum interval date : ', maximum_interval_train)
print('End interval date of training set :', end_interval_train2)

print("############################################################")
print("VALIDATION Fold 2")
print('Start interval date of testing set: ', start_interval_val2)
print('End interval date of testing set :', end_interval_val2)
print("############################################################")



print("############################################################")
print("TESTING Fold 2")
print('Start interval date of testing set: ', start_interval_test2)
print('End interval date of testing set :', end_interval_test2)
print("############################################################")

# Train Fold 1 = 1982-2019, Val = 2018-2020, Test = 2019 - 2021

# Train Fold 2 = 1982-2022, Val = 2021-2023, Test = 2022 - 2024

############################################################
TRAINING Fold 1
Start interval date of training set:  1982-01-01 00:00:00
End interval date of training set : 2019-12-01 00:00:00
############################################################
VALIDATION Fold 1
Start interval date of testing set:  2018-01-01 00:00:00
End interval date of testing set : 2020-12-01 00:00:00
############################################################
############################################################
TESTING Fold 1
Start interval date of testing set:  2019-01-01 00:00:00
End interval date of testing set : 2021-12-01 00:00:00
############################################################
############################################################
TRAINING Fold 2
Start interval date of training set:  1982-01-01 00:00:00
End interval date of training set : 2022-01-01 00:00:00
############################################################
VALIDATION Fold 2
Start interval date of testing set:  2

In [33]:
#Split test and train

Train_df_list1, Val_df_list1, Test_df_list1,Train_df_list2,  Val_df_list2 ,Test_df_list2 = [], [],[],[],[],[]
for i_table in range(len(list_table)):
  Train_df_list1.append(list_table[i_table][start_interval_train1:end_interval_train1])
  Val_df_list1.append(list_table[i_table][start_interval_val1:end_interval_val1])
  Test_df_list1.append(list_table[i_table][start_interval_test1:end_interval_test1])
  Train_df_list2.append(list_table[i_table][start_interval_train1:end_interval_train2])
  Val_df_list2.append(list_table[i_table][start_interval_val2:end_interval_val2])
  Test_df_list2.append(list_table[i_table][start_interval_test2:end_interval_test2])

fold =[]
fold.append([Train_df_list1,Test_df_list1,Val_df_list1])
fold.append([Train_df_list2,Test_df_list2,Val_df_list2])


In [35]:
# สมมติ list_table มีโครงสร้างแบบนี้
# list_table[-1] = Y (rainfall)
# list_table[:-1] = X (features ทั้งหมด)

# ---- Fold 1 ----
# X_train: features ทุกตัว ช่วง train
X_train_fold1 = pd.concat(Train_df_list1[:-1], axis=1)

# y_train: target (rainfall) ช่วง train
y_train_fold1 = Train_df_list1[-1]

# X_val
X_val_fold1   = pd.concat(Val_df_list1[:-1], axis=1)
y_val_fold1   = Val_df_list1[-1]

# X_test
X_test_fold1  = pd.concat(Test_df_list1[:-1], axis=1)
y_test_fold1  = Test_df_list1[-1]


In [34]:
Train_df_list1

[               MEIV2  RMM_AMPLITUDE     PHASE    BSISO1  BSISO1-Phase  \
 DATE                                                                    
 1982-01-01 -0.187714       0.714459 -1.386540 -0.948166     -0.832762   
 1982-02-01 -0.102040       0.281839  1.174365 -0.948166     -0.832762   
 1982-03-01 -1.056691       0.443087  0.747548 -0.948166     -0.832762   
 1982-04-01 -0.261148       0.738476  1.601183 -0.948166     -0.832762   
 1982-05-01 -0.579365      -0.930963 -0.532905  0.774939     -0.170381   
 ...              ...            ...       ...       ...           ...   
 2019-08-01 -0.322344      -0.930963 -0.106088  1.268179      1.154380   
 2019-09-01 -0.518170      -0.930963 -1.386540  0.932615      0.823190   
 2019-10-01 -0.334583      -0.930963 -0.959723  0.717530      1.816761   
 2019-11-01 -0.138757       1.058389  0.320730 -0.948166     -0.832762   
 2019-12-01 -0.273387       0.323061 -0.959723 -0.948166     -0.832762   
 
                  DMI       ONI     

In [37]:
print(f"Shape of X_train: {X_train_fold1.shape}")
print(f"Shape of X_test: {X_test_fold1.shape}")
print(f"Shape of y_train: {y_train_fold1.shape}")
print(f"Shape of y_test: {y_test_fold1.shape}")
print(f"Shape of X_val: {X_val_fold1.shape}")
print(f"Shape of y_val: {y_val_fold1.shape}")

Shape of X_train: (456, 3537)
Shape of X_test: (36, 3537)
Shape of y_train: (456, 27)
Shape of y_test: (36, 27)
Shape of X_val: (36, 3537)
Shape of y_val: (36, 27)


In [39]:
X_train_fold1.head()

,MEIV2,RMM_AMPLITUDE,PHASE,BSISO1,BSISO1-Phase,DMI,ONI,PDO,SWM,NE,...,ONI_lag1,ONI_lag2,DMI_lag1,DMI_lag2,BSISO1_lag1,BSISO1-Phase_lag1,SWM_lag1,Month_sin,Month_cos,581301
DATE,,,,,,,,,,,,,,,,,,,,,
1982-01-01,-0.187714,0.714459,-1.386540,-0.948166,-0.832762,-0.023677,-0.140501,0.366424,1.469381,1.793781,...,-0.324733,-0.452551,-0.523783,-0.532089,-0.948166,-0.832762,1.136801,0.500000,8.660254e-01,-1.147741
1982-02-01,-0.102040,0.281839,1.174365,-0.948166,-0.832762,1.265857,-0.042981,0.316895,0.837625,1.758152,...,-0.140014,-0.324464,-0.021566,-0.522618,-0.948166,-0.832762,1.466406,0.866025,5.000000e-01,-1.145926
1982-03-01,-1.056691,0.443087,0.747548,-0.948166,-0.832762,0.981589,-0.074650,0.177455,0.424145,1.158306,...,-0.042503,-0.139761,1.268030,-0.020539,-0.948166,-0.832762,0.835332,1.000000,6.123234e-17,-0.810033
1982-04-01,-0.261148,0.738476,1.601183,-0.948166,-0.832762,1.140032,0.200470,-0.098664,0.264021,1.416414,...,-0.074169,-0.042259,0.983748,1.268705,-0.948166,-0.832762,0.422298,0.866025,-5.000000e-01,-0.277445
1982-05-01,-0.579365,-0.930963,-0.532905,0.774939,-0.170381,1.531242,0.516375,-0.382280,-0.468001,0.380173,...,0.200925,-0.073923,1.142199,0.984501,-0.948166,-0.832762,0.262348,0.500000,-8.660254e-01,0.315058


# Feature selection

In [40]:
print("The first feature size that is climate index:", n_feature_climate)

The first feature size that is climate index: 26


focus แค่ climate

In [56]:
selected_index =[]
feature_cluster =  feature[feature['Cluster']==cluster]
for feature_index in range(len(feature_cluster)):
    for i in range(len(feature_climate)):
        if((feature_climate[i] == feature_cluster.iloc[feature_index,1]) and feature_cluster.iloc[feature_index,5] == 0):
                selected_index.append(i)

        elif(feature_cluster.iloc[feature_index,5] != 0):
            for i_lag in range(len(feature_climate)):
                if(feature_climate[i_lag] == feature_cluster.iloc[feature_index,1]+'_lag'+str(feature_cluster.iloc[feature_index,5])):
                    selected_index.append(i_lag)
                    break
                else:
                    continue  # Continue if the inner loop wasn't broken.
            break  # Inner loop was broken, break the outer.

selected_index.sort()
feature_climate[selected_index]


Index(['RMM_AMPLITUDE', 'BSISO1', 'DMI', 'SWM', 'NE', 'ONI_lag1', 'Month_sin',
       'Month_cos'],
      dtype='object')

In [59]:
random.seed(10)
position_optimal = [0 for i in range(n_feature_climate)]
#position_remain = (Train_df_list[0].shape[1] - n_feature_climate)*[1]
position_remain = [1] # select rain

for id in selected_index:
  position_optimal[id]=1

position_all = position_optimal + position_remain
position_all_index = [index for index,value in enumerate(position_all) if value == 1]
climate_name_list = [feature_climate[index] for index,value in enumerate(position_optimal) if value == 1]

print("position_all_index : ", position_all_index)
print("Climate index : ", climate_name_list)

position_all_index :  [1, 3, 5, 8, 9, 17, 24, 25, 26]
Climate index :  ['RMM_AMPLITUDE', 'BSISO1', 'DMI', 'SWM', 'NE', 'ONI_lag1', 'Month_sin', 'Month_cos']


In [47]:
def select_feature_f(list_df,position_feature_index):
    for i in range(len(list_df)):
        list_df[i] = list_df[i].iloc[:,position_feature_index]
    return list_df

In [48]:
#Fold 1 = Fold[0] , Fold 2 = Fold[1]
#Test Fold 1 = Fold[0][0] , Train Fold 1 = Fold[0][1], Validation Fold 1 = Fold[0][2]
fold[0][0] = select_feature_f(list_df = fold[0][0], position_feature_index = position_all_index)
fold[0][1] = select_feature_f(list_df = fold[0][1], position_feature_index = position_all_index)
fold[0][2] = select_feature_f(list_df = fold[0][2], position_feature_index = position_all_index)
fold[1][0] = select_feature_f(list_df = fold[1][0], position_feature_index = position_all_index)
fold[1][1] = select_feature_f(list_df = fold[1][1], position_feature_index = position_all_index)
fold[1][2] = select_feature_f(list_df = fold[1][2], position_feature_index = position_all_index)

In [49]:
len(Y_rainfall_normalized_df.columns)

132

In [50]:
cluster_station= []
for index in range(0,len(Y_rainfall_normalized_df.columns)):
    if (node['quality'].iloc[index]==1 and node['cluster'].iloc[index]==cluster) :
            cluster_station.append(index)


In [51]:
for index in range(len(cluster_station)):
    print(node['Node'].iloc[cluster_station[index]])

351201
376201
376202
376401
378201
379201
379401


In [52]:
num_feature = len(feature[feature['Cluster']==cluster])

In [53]:

def extract_features(feature,j, window_size,horizon,data_list):
    xs_list= []
    for station in cluster_station:
        xs_new= np.array(data_list[station].iloc[:,feature][j:j+window_size])
        xs_list.append([xs_new])
    xs = np.array(xs_list)
    return xs

In [54]:
climate_name_list

['RMM_AMPLITUDE',
 'BSISO1',
 'DMI',
 'SWM',
 'NE',
 'ONI_lag1',
 'Month_sin',
 'Month_cos']

In [55]:
num_feature

8